# M8 — Luồng B: Score tin mới (load model, KHÔNG train lại)

**Mục tiêu:** mỗi lô crawl → bảng dự đoán + danh sách top định giá thấp, không train lại.

**Kiến trúc:** nạp 1 artifact `PipelineModel` (M7, đã gồm feature stages M6 + RF) →
`transform` lô tin mới (đã ghép vĩ mô theo `Ngay dang`) → `prediction` = giá dự đoán (triệu/m²).

**Quyết định (user chốt 2026-09-17):**
- Lô demo = **toàn bộ `listings_clean/sale` (2156 dòng)** — chưa có crawl mới, tái dùng data hiện có.
- **Cờ định giá thấp:** `undervalued_ratio = (pred − thực)/pred ≥ 0.20` (20%).
- **Drift check:** RMSE lô mới vs `baseline_rmse` (M7). Cảnh báo "cần retrain" khi
  `rmse > 1.5 × baseline` (15.128 → **22.69**).
- ⚠️ **Trung thực:** RMSE trên full 2156 **lạc quan** (lô trùng data đã train) → drift check
  tính RMSE **chỉ trên 381 dòng test** (`randomSplit` seed=42 y hệt M7) để có số holdout thật.
  Bảng dự đoán vẫn xuất đủ full batch.

**Chạy lại:**
`JAVA_HOME=/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home .venv/bin/jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=python3 ml/score_new.ipynb`


In [1]:
import os, json, datetime
if os.path.basename(os.getcwd()) == "ml":
    os.chdir("..")
os.environ.setdefault(
    "JAVA_HOME",
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
)

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator

# --- Cấu hình (đổi NEW_BATCH khi có crawl mới) ---
NEW_BATCH  = "data/lake/listings_clean/sale"      # lô tin mới (M5); production: output crawl mới
MACRO      = "data/raw_csv/macro/macro_daily.csv"
VERSION    = "2026-09-17"
MODEL_PATH = f"models/v{VERSION}/model"            # FULL PipelineModel (M7)
METRICS    = f"models/v{VERSION}/metrics.json"
PRED_OUT   = "data/lake/predictions/sale"          # output: bảng dự đoán (M9 đọc)
LABEL      = "price_per_m2"                          # giá thực (triệu/m²) — có trong lô để drift check

UNDERVALUED_TH = 0.20     # cờ định giá thấp khi ratio >= 20%
DRIFT_MULT     = 1.5      # cảnh báo retrain khi rmse > 1.5 * baseline
SEED           = 42       # y hệt M7 để tái tạo test split cho drift

with open(METRICS) as f:
    BASELINE_RMSE = json.load(f)["baseline_rmse"]
DRIFT_TH = DRIFT_MULT * BASELINE_RMSE
print(f"baseline_rmse={BASELINE_RMSE:.3f} | drift threshold={DRIFT_TH:.3f} (>{DRIFT_MULT}x)")

spark = (
    SparkSession.builder.appName("M8-score")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

baseline_rmse=15.128 | drift threshold=22.691 (>1.5x)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/17 20:01:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# 1) Đọc lô tin mới + GHÉP VĨ MÔ theo Ngay dang (y hệt M6, model không tự join được)
listings = spark.read.parquet(NEW_BATCH)
print("lô mới:", listings.count(), "dòng,", len(listings.columns), "cột")

macro = (
    spark.read.option("header", True).csv(MACRO)
    .withColumn("d", F.to_date("date"))
    .withColumn("gold_usd", F.col("gold_usd").cast("double"))
    .withColumn("usdvnd",   F.col("usdvnd").cast("double"))
    .withColumn("vnindex",  F.col("vnindex").cast("double"))
    .select("d", "gold_usd", "usdvnd", "vnindex")
)
listings = listings.withColumn("posted_date", F.to_date("Ngay dang"))
feat = listings.join(macro, listings["posted_date"] == macro["d"], "left").drop("d")

miss = feat.filter(
    F.col("vnindex").isNull() | F.col("gold_usd").isNull() | F.col("usdvnd").isNull()
).count()
if miss:
    print(f"⚠ {miss} dòng ngoài cửa sổ macro (null vĩ mô) — mở rộng fetch_macro --start; "
          "VectorAssembler handleInvalid=error sẽ chặn các dòng này")
else:
    print("macro coverage: đủ, 0 dòng thiếu")

lô mới: 2156 dòng, 23 cột


macro coverage: đủ, 0 dòng thiếu


In [3]:
# 2) Nạp FULL PipelineModel + transform → prediction (giá dự đoán triệu/m²)
model = PipelineModel.load(MODEL_PATH)
print("nạp model:", [type(s).__name__ for s in model.stages])

scored = model.transform(feat)

# 3) undervalued_ratio + cờ. prediction = giá/m² dự đoán; LABEL = giá/m² thực
scored = (
    scored
    .withColumnRenamed("prediction", "predicted_ppm2")
    .withColumn("undervalued_ratio",
                (F.col("predicted_ppm2") - F.col(LABEL)) / F.col("predicted_ppm2"))
    .withColumn("is_undervalued", F.col("undervalued_ratio") >= F.lit(UNDERVALUED_TH))
    # giá tổng (VND) cho dễ đọc: giá/m² (triệu) * diện tích * 1e6
    .withColumn("predicted_total_vnd",
                F.round(F.col("predicted_ppm2") * F.col("Dien tich") * 1e6).cast("long"))
)
n_scored = scored.count()
n_under  = scored.filter("is_undervalued").count()
print(f"đã chấm {n_scored} tin | định giá thấp (>={int(UNDERVALUED_TH*100)}%): {n_under} "
      f"({100*n_under/n_scored:.1f}%)")

nạp model: ['SQLTransformer', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'StandardScalerModel', 'RandomForestRegressionModel']


26/09/17 20:01:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


đã chấm 2156 tin | định giá thấp (>=20%): 317 (14.7%)


In [4]:
# 4) DRIFT CHECK — RMSE chỉ trên 381 dòng test (randomSplit seed=42 y hệt M7)
#    (full batch trùng data train -> RMSE lạc quan; test split = holdout thật)
_, test = feat.randomSplit([0.8, 0.2], seed=SEED)
test_pred = model.transform(test)
ev = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="rmse")
batch_rmse = ev.evaluate(test_pred)

# RMSE trên full batch (tham khảo, sẽ thấp hơn vì gồm cả train)
full_rmse = RegressionEvaluator(
    labelCol=LABEL, predictionCol="predicted_ppm2", metricName="rmse"
).evaluate(scored)

drift = batch_rmse > DRIFT_TH
print(f"RMSE test-holdout = {batch_rmse:.3f} | baseline = {BASELINE_RMSE:.3f} "
      f"| ngưỡng drift = {DRIFT_TH:.3f}")
print(f"RMSE full-batch   = {full_rmse:.3f} (lạc quan — gồm data train)")
if drift:
    print(f"🔴 DRIFT: RMSE {batch_rmse:.3f} > {DRIFT_TH:.3f} → CẦN RETRAIN (chạy lại M7)")
else:
    print(f"🟢 OK: RMSE {batch_rmse:.3f} <= {DRIFT_TH:.3f} → không cần retrain")

RMSE test-holdout = 15.128 | baseline = 15.128 | ngưỡng drift = 22.691
RMSE full-batch   = 11.052 (lạc quan — gồm data train)
🟢 OK: RMSE 15.128 <= 22.691 → không cần retrain


In [5]:
# 5) Bảng dự đoán (cột cho M9) + ghi parquet + top định giá thấp
SCORED_AT = datetime.datetime.now().isoformat(timespec="seconds")
preds = (
    scored.select(
        F.col("STT").alias("id"),
        F.col("Quan/Huyen").alias("district"),
        F.col("Dien tich").alias("area_m2"),
        F.round(F.col(LABEL), 2).alias("listing_ppm2"),          # giá thực (triệu/m²)
        F.round(F.col("predicted_ppm2"), 2).alias("predicted_ppm2"),
        F.col("predicted_total_vnd"),
        F.round(F.col("undervalued_ratio"), 4).alias("undervalued_ratio"),
        F.col("is_undervalued"),
        F.col("URL"),
        F.lit(VERSION).alias("model_version"),
        F.lit(SCORED_AT).alias("scored_at"),
    )
)
preds.write.mode("overwrite").parquet(PRED_OUT)
print(f"ghi bảng dự đoán -> {PRED_OUT} | {preds.count()} dòng")

print("\n=== TOP 15 ĐỊNH GIÁ THẤP (undervalued_ratio giảm dần) ===")
(preds.filter("is_undervalued")
      .orderBy(F.col("undervalued_ratio").desc())
      .select("district", "area_m2", "listing_ppm2", "predicted_ppm2",
              "undervalued_ratio", "predicted_total_vnd", "URL")
      .show(15, truncate=60))

print("=== Số tin định giá thấp theo quận ===")
(preds.filter("is_undervalued").groupBy("district").count()
      .orderBy(F.col("count").desc()).show(25, truncate=False))

print("=" * 55)
print("M8 XONG:")
print(f"  lô chấm         : {NEW_BATCH} ({n_scored} tin)")
print(f"  định giá thấp   : {n_under} tin (>={int(UNDERVALUED_TH*100)}%)")
print(f"  drift (test)    : RMSE={batch_rmse:.3f} vs baseline={BASELINE_RMSE:.3f} "
      f"-> {'RETRAIN' if drift else 'OK'}")
print(f"  bảng dự đoán    : {PRED_OUT}")
spark.stop()

ghi bảng dự đoán -> data/lake/predictions/sale | 2156 dòng

=== TOP 15 ĐỊNH GIÁ THẤP (undervalued_ratio giảm dần) ===


+-----------------+-------+------------+--------------+-----------------+-------------------+------------------------------------+
|         district|area_m2|listing_ppm2|predicted_ppm2|undervalued_ratio|predicted_total_vnd|                                 URL|
+-----------------+-------+------------+--------------+-----------------+-------------------+------------------------------------+
|           Quận 7|  125.0|        16.0|         73.53|           0.7824|         9191303309|https://www.chotot.com/129848717.htm|
|Thành phố Thủ Đức|   62.0|       10.97|          43.2|           0.7461|         2678504662|https://www.chotot.com/129327608.htm|
|Thành phố Thủ Đức|   60.0|       14.17|         43.36|           0.6733|         2601722688|https://www.chotot.com/129094642.htm|
|           Quận 7|   68.0|       22.06|         65.03|           0.6608|         4422047217|https://www.chotot.com/129730006.htm|
|   Quận Phú Nhuận|   80.0|       18.75|         52.26|           0.6412|         4

+-----------------+-----+
|district         |count|
+-----------------+-----+
|Thành phố Thủ Đức|65   |
|Quận 7           |31   |
|Huyện Bình Chánh |27   |
|Quận Bình Thạnh  |25   |
|Quận 8           |22   |
|Quận Bình Tân    |21   |
|Quận Gò Vấp      |19   |
|Quận 5           |18   |
|Quận 6           |17   |
|Quận Tân Phú     |17   |
|Quận 12          |15   |
|Quận 10          |10   |
|Quận Tân Bình    |8    |
|Quận 11          |6    |
|Huyện Hóc Môn    |5    |
|Huyện Nhà Bè     |5    |
|Quận 4           |4    |
|Quận 1           |1    |
|Quận Phú Nhuận   |1    |
+-----------------+-----+

M8 XONG:
  lô chấm         : data/lake/listings_clean/sale (2156 tin)
  định giá thấp   : 317 tin (>=20%)
  drift (test)    : RMSE=15.128 vs baseline=15.128 -> OK
  bảng dự đoán    : data/lake/predictions/sale
